# Short Pick v3 R14 后续优化实验

## tl;dr

- **保留 R14，不替换策略。** 两组候选均未通过九指标不劣化门槛。
- **2025-10 并非 R14 亏损月。** 统一重建口径下该月收益为 **+1.32%**；加入 10 日市场动量确认后降至 **+0.57%**，并把负月从 2 个扩大到 5 个。
- **弱市降权只有局部防守价值。** 最温和方案改善回撤和最差月，但总收益减少约 7.57 个百分点，负月增加到 3 个。
- **晋级仍被历史执行快照缺失阻断。** Top3 选择 1,533/1,533 精确复现，但重建 R14 总收益为 332.10%，低于静态合同 341.77%。

## Context & Methods

决策问题：是否应以 2025-10 为切入点替换当前 R14 优化前沿。

### Key Assumptions

- 比较窗口固定为 2023-09-07 至 2026-06-26，初始资金 20 万元。
- 买入、费用、100 股整手、滚动 tranche、替补和 25% 市值再平衡使用同一逐订单内核。
- 只测试两个事前定义的规则族：短周期市场确认、弱基准降权。
- 原执行数据库快照已按存储生命周期清理，因此结果只能支持方向性淘汰，不能支持晋级。

## Data

加载提交内的紧凑实验制品；重型 PIT 特征矩阵和运行库只作为源引用，不嵌入 notebook。

In [1]:
import json
from pathlib import Path

artifact_path = Path("docs/contracts/SHORTPICK_V3_R14_OCTOBER_OPTIMIZATION_EXPERIMENT_2026-07-15.json")
artifact = json.loads(artifact_path.read_text(encoding="utf-8"))
assert artifact["selection_validation"]["passed"]
assert artifact["bar_reconstruction_validation"]["passed"]
artifact["status"], artifact["decision"], artifact["promotion_blocking_gate_ids"]

('completed_research_only_reproduction_gap',
 'retain_r14_no_candidate_cleared_replacement_gate',
 ['r14_contract_reproduction_mismatch'])

In [2]:
import pandas as pd

rows = []
for variant in artifact["variants"]:
    summary = variant["summary"]
    rows.append({
        "config_id": variant["config_id"],
        "total_return_pct": summary["total_return"] * 100,
        "annualized_return_pct": summary["annualized_return"] * 100,
        "max_drawdown_pct": summary["max_drawdown"] * 100,
        "negative_month_count": summary["negative_month_count"],
        "worst_month_pct": summary["worst_monthly_return"] * 100,
        "skipped_order_rate_pct": summary["skipped_order_rate"] * 100,
        "october_2025_return_pct": variant["october_2025_return"] * 100,
        "directional_all_nine_non_degraded": variant["directional_vs_reconstructed_baseline"]["all_nine_metrics_non_degraded"],
        "directional_passed": variant["directional_vs_reconstructed_baseline"]["passed"],
    })
variants = pd.DataFrame(rows)
variants.round(3)

,config_id,total_return_pct,annualized_return_pct,max_drawdown_pct,negative_month_count,worst_month_pct,skipped_order_rate_pct,october_2025_return_pct,directional_all_nine_non_degraded,directional_passed
0,daily_15_tranche_rank_adjusted_r5_093_strong15...,332.100,68.627,-6.885,2,-1.413,15.145,1.320,True,False
1,r14_strong_benchmark10_ge_m0p01_research_v1,332.100,68.627,-6.885,2,-1.413,15.145,1.320,True,False
2,r14_strong_benchmark10_ge_0p0_research_v1,332.100,68.627,-6.885,2,-1.413,15.145,1.320,True,False
3,r14_strong_benchmark10_ge_0p01_research_v1,320.188,66.953,-7.234,5,-1.413,12.967,0.567,False,False
4,r14_strong_benchmark10_ge_0p02_research_v1,320.188,66.953,-7.234,5,-1.413,12.967,0.567,False,False
5,r14_weak_benchmark20_lt_m0p02_scale_0p9_resear...,324.529,67.566,-6.704,3,-1.313,14.627,1.465,False,False
6,r14_weak_benchmark20_lt_m0p02_scale_0p8_resear...,305.195,64.801,-6.760,3,-1.459,14.108,1.376,False,False
7,r14_weak_benchmark20_lt_0p0_scale_0p9_research_v1,316.607,66.443,-6.739,2,-1.339,14.627,1.287,False,False


## Results

下面独立复算相对重建基线的关键变化，避免直接复述生成器的结论字段。

In [3]:
baseline = variants.iloc[0]
comparison = variants.iloc[1:].copy()
for metric in ["total_return_pct", "max_drawdown_pct", "negative_month_count", "worst_month_pct", "october_2025_return_pct"]:
    comparison[f"delta_{metric}"] = comparison[metric] - baseline[metric]
comparison[[
    "config_id",
    "delta_total_return_pct",
    "delta_max_drawdown_pct",
    "delta_negative_month_count",
    "delta_worst_month_pct",
    "delta_october_2025_return_pct",
    "directional_all_nine_non_degraded",
]].round(3)

,config_id,delta_total_return_pct,delta_max_drawdown_pct,delta_negative_month_count,delta_worst_month_pct,delta_october_2025_return_pct,directional_all_nine_non_degraded
1,r14_strong_benchmark10_ge_m0p01_research_v1,0.000,0.000,0,0.000,0.000,True
2,r14_strong_benchmark10_ge_0p0_research_v1,0.000,0.000,0,0.000,0.000,True
3,r14_strong_benchmark10_ge_0p01_research_v1,-11.912,-0.349,3,0.000,-0.753,False
4,r14_strong_benchmark10_ge_0p02_research_v1,-11.912,-0.349,3,0.000,-0.753,False
5,r14_weak_benchmark20_lt_m0p02_scale_0p9_resear...,-7.571,0.181,1,0.100,0.145,False
6,r14_weak_benchmark20_lt_m0p02_scale_0p8_resear...,-26.905,0.125,1,-0.046,0.056,False
7,r14_weak_benchmark20_lt_0p0_scale_0p9_research_v1,-15.493,0.146,0,0.074,-0.033,False


In [4]:
expected = artifact["baseline_reproduction"]["checks"]["total_return"]["expected"]
observed = artifact["baseline_reproduction"]["checks"]["total_return"]["observed"]
selection = artifact["selection_validation"]
{
    "expected_contract_total_return_pct": round(expected * 100, 3),
    "reconstructed_total_return_pct": round(observed * 100, 3),
    "gap_percentage_points": round((observed - expected) * 100, 3),
    "top3_selection_matches": f"{selection['observed_count'] - selection['mismatch_count']}/{selection['expected_count']}",
}

{'expected_contract_total_return_pct': 341.767,
 'reconstructed_total_return_pct': 332.1,
 'gap_percentage_points': -9.667,
 'top3_selection_matches': '1533/1533'}

## Takeaways

1. **停止沿 2025-10 做防守调参。** 该月本身盈利，短周期确认会错杀有效强信号。
2. **弱市降权不进入策略集。** 它改善部分风险指标，但无法同时守住收益和负月门槛。
3. **下一步先恢复可晋级的执行快照合同。** 在新的冻结行情快照上复跑 R14，并把快照 ID、紧凑订单摘要和重建命令一起保留。
4. **恢复后优先研究执行效率，不再新增规则族。** 目标应是减少 58 次替补与现金闲置的摩擦，同时保持信号权重不变；任何候选必须一进一出替换 R14。